# Revisión capa BRONCE (datos crudos)

Inventario de los datos originales de la NYC TLC tal cual se descargaron.
Esta capa es **inmutable**: nada del pipeline la modifica.

Se espera: 144 archivos (4 categorías × 12 meses × 3 años: 2023–2025)
+ zone lookup + auditoría.

In [ ]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

## Inventario por categoría

In [ ]:
resumen = []
for cat in CATS:
    d = DATA / "bronze" / cat
    archivos = sorted(d.glob("*.parquet"))
    resumen.append({
        "categoria": cat,
        "archivos": len(archivos),
        "filas": parquet_rows(d),
        "peso_mb": dir_mb(d),
    })
inv = pl.DataFrame(resumen)
print(f"TOTAL bronce: {inv['filas'].sum():,} filas | {inv['peso_mb'].sum():,.0f} MB")
inv

## Detalle mensual (filas por archivo, 36 meses)

In [ ]:
detalle = []
for cat in CATS:
    for y, m in MESES:
        f = DATA / "bronze" / cat / f"{y}-{m:02d}.parquet"
        detalle.append({"categoria": cat, "mes": f"{y}-{m:02d}", "filas": parquet_rows(f)})
(pl.DataFrame(detalle)
   .pivot(values="filas", index="mes", on="categoria")
   .sort("mes"))

## Tabla de zonas (zone lookup)

In [ ]:
zonas = pl.read_parquet(DATA / "bronze" / "zone-lookup" / "zone-lookup-table.parquet")
print(f"{zonas.height} zonas | boroughs: {sorted(zonas['Borough'].unique().to_list())}")
zonas.head(5)

## Auditoría de descargas

In [ ]:
audit_b = pl.read_parquet(DATA / "bronze" / "audit.parquet")
print(f"{audit_b.height} registros de descarga | filas totales auditadas: {audit_b['rowcount'].sum():,}")
audit_b.select("name", "rowcount", "bytecount", "end_timestamp").tail(10)